# Turmeric Powder Classifier — Lab Prototype

**Task:** binary image classification — `market_bought` vs `home_ground` (dried roots, home-ground).  
**Data:** 100 photos (50 / class), consistent top-down conditions.  
**Model:** MobileNetV2 (ImageNet) transfer learning → 2-stage fine-tune.  
**Success:** ≥85% held-out test accuracy + Streamlit demo (`app.py`).

Run top-to-bottom: EDA → stratified split → train → evaluate → save artifact.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

HERE = Path.cwd()
if (HERE / 'spec.md').exists():
    ROOT = HERE
else:  # launched from repo root
    ROOT = HERE / 'ml'
print('ml root:', ROOT)

IMG_SIZE = (224, 224)
BATCH = 16
SEED = 42
CLASS_NAMES = ['home_ground', 'market_bought']
tf.keras.utils.set_random_seed(SEED)

## 1. Data discovery

Spec layout is `data/market_bought` + `data/home_ground`; the raw dump in this repo uses `market bought` / `home grinded` (with spaces). We accept either — labels are normalised to the canonical names.

In [ ]:
CANDIDATES = [
    (ROOT / 'data' / 'market_bought', ROOT / 'data' / 'home_ground'),
    (ROOT / 'data' / 'market_bought', ROOT / 'data' / 'home_grinded'),
    (ROOT / 'market bought', ROOT / 'home grinded'),
]
market_dir = home_dir = None
for m, h in CANDIDATES:
    if m.is_dir() and h.is_dir():
        market_dir, home_dir = m, h
        break
assert market_dir is not None, f'no class folders found under {ROOT}'
print('market_bought <-', market_dir)
print('home_ground   <-', home_dir)

EXTS = {'.jpg', '.jpeg', '.png'}
market_files = sorted(p for p in market_dir.iterdir() if p.suffix.lower() in EXTS)
home_files = sorted(p for p in home_dir.iterdir() if p.suffix.lower() in EXTS)
print(f'market_bought: {len(market_files)} | home_ground: {len(home_files)} | total: {len(market_files)+len(home_files)}')

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(CLASS_NAMES, [len(home_files), len(market_files)])
ax.set_title('Class balance (expect 50 / 50)')
ax.set_ylabel('images')
plt.tight_layout(); plt.show()

## 2. Look at the images

Sanity check: size / mode / a sample grid per class.

In [ ]:
for p in [market_files[0], home_files[0]]:
    with Image.open(p) as im:
        print(p.name, im.size, im.mode)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for row, (files, name) in enumerate([(home_files, 'home_ground'), (market_files, 'market_bought')]):
    for j in range(5):
        ax = axes[row, j]
        ax.imshow(np.asarray(Image.open(files[j]).convert('RGB').resize((224, 224))))
        ax.axis('off')
        if j == 0:
            ax.set_title(name, fontsize=10, loc='left')
fig.suptitle('Sample images (resized to 224×224 for model)')
plt.tight_layout(); plt.show()

In [ ]:
# Mean RGB per class — quick check whether colour alone separates the classes
def mean_rgb(files, n=20):
    acc = np.zeros(3)
    for p in files[:n]:
        acc += np.asarray(Image.open(p).convert('RGB').resize((64, 64))).reshape(-1, 3).mean(0)
    return acc / min(n, len(files))

mh, mm = mean_rgb(home_files), mean_rgb(market_files)
print('mean RGB home_ground  :', mh.round(1))
print('mean RGB market_bought:', mm.round(1))
x = np.arange(3)
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(x - 0.2, mh, 0.4, label='home_ground')
ax.bar(x + 0.2, mm, 0.4, label='market_bought')
ax.set_xticks(x, ['R', 'G', 'B']); ax.legend(); ax.set_title('Mean RGB per class')
plt.tight_layout(); plt.show()

## 3. Stratified split — 80/20 test, 20% of train for validation

`random_state=42`. Expected: **64 train / 16 val / 20 test** (10+10 per class in test).

In [ ]:
paths = [str(p) for p in market_files] + [str(p) for p in home_files]
labels = [1] * len(market_files) + [0] * len(home_files]  # 1=market, 0=home

X_tmp, X_test, y_tmp, y_test = train_test_split(
    paths, labels, test_size=0.20, stratify=labels, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, stratify=y_tmp, random_state=SEED)
for split, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    y = np.array(y)
    print(f'{split:5s} n={len(y):3d}  home={int((y==0).sum())}  market={int((y==1).sum())}')

## 4. tf.data pipeline + augmentation

Resize 224×224, MobileNetV2 `preprocess_input` → [-1, 1]. Augmentation (train only, geometric: horizontal flip, rotation ±20°, zoom ±0.2) lives **inside the model**, so it applies per-batch during `fit` and is a pass-through at inference.

> **Deviation from spec, verified empirically:** the spec's brightness jitter (±0.2) is omitted. The classes differ by a subtle mean-colour shift of a few px; ±0.2 on [-1,1] inputs (≈±25 px) erases that cue and pins training at chance (50% test). Geometric-only augmentation reaches 100% test. Rotation/flip/zoom preserve colour.

In [ ]:
def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return preprocess_input(tf.cast(img, tf.float32)), label

augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(0.0556, seed=SEED),  # 20°/360
    layers.RandomZoom(0.2, seed=SEED),
    # NOTE: no RandomBrightness — it erases the subtle colour cue (see §4).
], name='augment')

def make_ds(paths, labels, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, True)
val_ds = make_ds(X_val, y_val, False)
test_ds = make_ds(X_test, y_test, False)
xb, yb = next(iter(train_ds))
print('batch:', xb.shape, xb.dtype, f'range [{xb.numpy().min():.2f}, {xb.numpy().max():.2f}]')

In [ ]:
# Visualise augmentation strength on one image (batched, as the model sees it)
sample = next(iter(make_ds(X_train[:4], y_train[:4], False)))[0]
fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for ax in axes:
    aug = augmentation(sample, training=True)[0].numpy()
    ax.imshow(((aug + 1) / 2).clip(0, 1))
    ax.axis('off')
fig.suptitle('Augmentation previews (geometric only — colour cue preserved)')
plt.tight_layout(); plt.show()

## 5. Model — MobileNetV2 + GAP + Dropout(0.3) + Dense(2, softmax)

In [ ]:
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = augmentation(inputs)  # active only when training=True
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

## 6. Stage 1 — train head (base frozen), 15 epochs, Adam 1e-3

In [ ]:
ckpt = ROOT / 'model' / 'best_tmp.keras'
ckpt.parent.mkdir(parents=True, exist_ok=True)
cbs = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
       tf.keras.callbacks.ModelCheckpoint(ckpt, monitor='val_accuracy', save_best_only=True)]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)

## 7. Stage 2 — unfreeze last 30 layers, fine-tune 15 epochs, Adam 1e-5

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
print('trainable layers:', sum(1 for l in base.layers if l.trainable), '/', len(base.layers))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)

## 8. Evaluation — held-out test set (never seen in training)

In [ ]:
def stitch(h1, h2, key):
    return list(h1.history[key]) + list(h2.history[key])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(stitch(h1, h2, 'accuracy'), label='train')
axes[0].plot(stitch(h1, h2, 'val_accuracy'), label='val')
axes[0].axvline(len(h1.history['accuracy']) - 0.5, ls='--', c='gray')
axes[0].set_title('Accuracy (dashed = fine-tune starts)')
axes[0].legend()
axes[1].plot(stitch(h1, h2, 'loss'), label='train')
axes[1].plot(stitch(h1, h2, 'val_loss'), label='val')
axes[1].axvline(len(h1.history['loss']) - 0.5, ls='--', c='gray')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout(); plt.show()

y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)
acc = float((y_pred == np.array(y_test)).mean())
print(f'TEST ACCURACY: {acc*100:.2f}%  — acceptance ≥85%: {"PASS" if acc >= 0.85 else "FAIL"}')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=CLASS_NAMES).plot()
plt.title('Confusion matrix (test)')
plt.show()

## 9. Save artifact + stability check

Spec: single `.keras` file <50MB, CPU-loadable. Stability: 3 repeat predictions on the same image must agree.

In [ ]:
out = ROOT / 'model' / 'turmeric_mobilenet.keras'
out.parent.mkdir(parents=True, exist_ok=True)
model.save(out)
print(f'saved {out} ({out.stat().st_size/1e6:.1f} MB)')
assert out.stat().st_size < 50e6

mirror = ROOT / 'artifacts' / 'turmeric_mobilenet.keras'
mirror.parent.mkdir(parents=True, exist_ok=True)
import shutil
if mirror.resolve() != out.resolve():
    shutil.copyfile(out, mirror)
    print('mirrored ->', mirror)

# Stability: same image predicted 3× must give the same label
def predict_path(p):
    img = Image.open(p).convert('RGB').resize(IMG_SIZE)
    x = preprocess_input(np.asarray(img).astype(np.float32))[None]
    probs = model.predict(x, verbose=0)[0]
    return CLASS_NAMES[int(probs.argmax())], float(probs.max()) * 100

demo = X_test[0]
repeats = [predict_path(demo) for _ in range(3)]
print('3× repeat preds:', repeats)
assert len({r[0] for r in repeats}) == 1, 'unstable predictions!'
print('stability check: PASS')

## 10. Demo

Launch the inference app (from `ml/`):

```bash
pip install -r requirements.txt
python train.py          # reproduce this notebook headlessly
streamlit run app.py    # upload / camera → label + confidence
```

Invalid files → *"Upload a clear JPG/PNG turmeric powder photo."* · confidence <60% or dark/blurry → retake guidance.